# JALARI — Visi AI: Latih Model Klasifikasi Limbah Organik MBG
Transfer learning **MobileNetV2** → **3 kelas**: `buah`, `sayur`, `daging`. *(Kelas `tulang` ditambahkan nanti — lihat catatan di bawah.)* Output langsung dipakai `public/js/vision.js` (TensorFlow.js, drop-in).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mostoples/jalari/blob/main/notebooks/jalari_vision_train.ipynb)

## Cara pakai (tanpa restart)
1. **Runtime → Change runtime type → GPU** (T4 gratis).
2. **Runtime → Run all.**
3. Saat diminta, upload **`kaggle.json`** (Kaggle → Settings → Create New API Token).

> **Kunci agar lancar:** Sel 1 menginstal `tensorflowjs` **sebelum** TensorFlow di-import sekali pun, sehingga TF dimuat pada versi yang konsisten dengan tensorflowjs. Ini mencegah error `undefined symbol` yang muncul kalau TF diturunkan versinya di tengah sesi. **Jangan menjalankan sel lain sebelum Sel 1 selesai.**

Di akhir, **`jalari_model.zip`** otomatis terunduh → ekstrak ke **`public/model/`** di repo JALARI.

In [ ]:
# 1) Install SEMUA dependency LEBIH DULU (sebelum import TensorFlow).
#    Urutan ini penting: tensorflowjs menetapkan versi TF, lalu TF dimuat sekali pada versi itu.
!pip install -q tensorflowjs split-folders kaggle

import tensorflow as tf            # import pertama -> versi konsisten dgn tensorflowjs
try:
    import tensorflowjs as tfjs
except ModuleNotFoundError as e:
    !pip install -q {e.name}
    import tensorflowjs as tfjs
print('TF', tf.__version__, '| tfjs', tfjs.__version__, '| GPU:', bool(tf.config.list_physical_devices('GPU')))
print('Sel 1 OK — lanjut Run all.')

In [ ]:
# 2) Upload kaggle.json (WAJIB)
import os
from google.colab import files
print('Pilih file kaggle.json ...')
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json
print('kaggle.json siap.')

In [ ]:
# 3) Unduh dataset: buah+sayur (Kritik Seth) & daging (Meat Quality Assessment)
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition -p /content/dl/fv --unzip -q
!kaggle datasets download -d crowww/meat-quality-assessment-based-on-deep-learning -p /content/dl/meat --unzip -q
import glob
print('fv  :', len(glob.glob('/content/dl/fv/**/*.jpg', recursive=True)), 'gambar')
print('meat:', len([p for p in glob.glob('/content/dl/meat/**/*', recursive=True) if p.lower().endswith(('.jpg','.jpeg','.png'))]), 'gambar')

In [ ]:
# 4) Susun dataset 3 kelas (buah, sayur, daging) + seimbangkan
import os, glob, shutil, random
random.seed(42)
FRUITS = {'apple','banana','grapes','kiwi','lemon','mango','orange','pear','pineapple','pomegranate','watermelon'}
EXT = ('.jpg','.jpeg','.png','.bmp','.webp')
def imgs(d): return [p for p in glob.glob(d+'/**/*', recursive=True) if p.lower().endswith(EXT)]
buckets = {'buah':[], 'sayur':[], 'daging':[]}
for p in imgs('/content/dl/fv'):
    cls = os.path.basename(os.path.dirname(p)).lower().strip()
    buckets['buah' if cls in FRUITS else 'sayur'].append(p)
buckets['daging'] = imgs('/content/dl/meat')

CAP = 800  # batas per kelas agar seimbang
OUT = '/content/dataset'; shutil.rmtree(OUT, ignore_errors=True)
classes = []
for c, ps in buckets.items():
    if len(ps) < 30:
        print(f'  lewati "{c}" (hanya {len(ps)} gambar)'); continue
    random.shuffle(ps); ps = ps[:CAP]
    os.makedirs(f'{OUT}/{c}', exist_ok=True)
    for i, p in enumerate(ps):
        ext = os.path.splitext(p)[1].lower() or '.jpg'
        shutil.copy(p, f'{OUT}/{c}/{c}_{i}{ext}')
    classes.append(c); print(f'  {c}: {len(ps)} gambar')
print('KELAS FINAL:', classes)
assert len(classes) >= 2, 'Butuh minimal 2 kelas.'

In [ ]:
# 5) Split train/val 85:15
import splitfolders
splitfolders.ratio('/content/dataset', output='/content/data', seed=42, ratio=(0.85, 0.15))
print('Split selesai.')

In [ ]:
# 6) Generator + bangun MobileNetV2 (base di-freeze) + latih
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
IMG, BATCH = 224, 32
tg = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input,
        rotation_range=20, zoom_range=0.2, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)
vg = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)
train = tg.flow_from_directory('/content/data/train', target_size=(IMG,IMG), batch_size=BATCH, class_mode='categorical')
val   = vg.flow_from_directory('/content/data/val',   target_size=(IMG,IMG), batch_size=BATCH, class_mode='categorical', shuffle=False)
NUM = train.num_classes
# PENTING: preprocessing [-1,1] ada di LUAR model (vision.js melakukannya). Jangan tambah layer Rescaling.
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG,IMG,3)); base.trainable = False
x = GlobalAveragePooling2D()(base.output); x = Dropout(0.3)(x); out = Dense(NUM, activation='softmax')(x)
model = Model(base.input, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
      tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]
model.fit(train, validation_data=val, epochs=20, callbacks=cb)

In [ ]:
# 7) (Opsional) Fine-tuning 30 layer teratas untuk dorong akurasi
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=val, epochs=5, callbacks=cb)

In [ ]:
# 8) Evaluasi
from sklearn.metrics import classification_report, confusion_matrix
labels = [k for k, _ in sorted(train.class_indices.items(), key=lambda x: x[1])]
p = model.predict(val); yp = p.argmax(1); yt = val.classes
print(classification_report(yt, yp, target_names=labels))
print('Confusion matrix:\n', confusion_matrix(yt, yp))

In [ ]:
# 9) Simpan model (.h5)
model.save('/content/jalari_vision.h5')
print('Model tersimpan.')

In [ ]:
# 10) Konversi ke TensorFlow.js (Python API + labels.json sesuai kontrak vision.js)
import json
labels = sorted([d for d in os.listdir('/content/data/train') if os.path.isdir('/content/data/train/'+d)])
print('labels =', labels)
os.makedirs('/content/model', exist_ok=True)
try:
    tfjs.converters.save_keras_model(model, '/content/model', quantization_dtype_map={'uint8': '*'})
except TypeError:
    tfjs.converters.save_keras_model(model, '/content/model')   # versi lama tanpa kuantisasi
json.dump(labels, open('/content/model/labels.json', 'w'))
print('Isi /content/model ->', os.listdir('/content/model'))

In [ ]:
# 11) Zip + unduh -> ekstrak ke public/model/ di repo JALARI
import shutil
shutil.make_archive('/content/jalari_model', 'zip', '/content/model')
from google.colab import files
files.download('/content/jalari_model.zip')

## Menyambungkan ke frontend (Fase 3 — sudah ada)
1. Ekstrak `jalari_model.zip` → salin `model.json`, `group1-shard*.bin`, `labels.json` ke **`public/model/`**.
2. Tes lokal: `cd public && python -m http.server 8000` → buka `http://localhost:8000/#/vision`. `vision.js` otomatis memakai model ini (berhenti memakai MobileNet demo).
3. Deploy: `firebase deploy --only hosting`.

## Menambahkan kelas `tulang` nanti
Kumpulkan 50–150 foto tulang → taruh di folder `/content/dataset/tulang/` **sebelum** Sel 5 (split), lalu Run all dari Sel 5. `Dense` otomatis jadi 4 kelas dan `labels.json` jadi `["buah","daging","sayur","tulang"]`. Frontend `vision.js` sudah siap 4 kelas.

**Kontrak kompatibilitas:** input 224×224 · preprocessing `[-1,1]` di luar model · `labels.json` = array urut indeks · output di `public/model/` (`model.json`, `group1-shard*.bin`, `labels.json`).